[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week1_data_foundations/day04_eda/day04_notebook.ipynb)

# Day 4 / 42: Exploratory Data Analysis (EDA)
**#42DaysOfML**

---

## What You Will Learn
- What EDA actually is and why it matters before any modelling
- Univariate analysis: understanding one column at a time
- Bivariate analysis: relationships between two columns
- Class imbalance detection (the accuracy paradox trap)
- Multivariate analysis: correlation heatmaps and pairplots
- Missing value visualisation
- How to generate findings, not just plots

**Dataset used:** Titanic (built-in via seaborn or synthetic recreation)

**Time to complete:** 45-60 minutes

---

## Step 0: Install and Import Libraries

In [ ]:
# Run this cell first if you are on Google Colab
# missingno is not pre-installed on Colab
!pip install missingno -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("All libraries loaded successfully")

## Step 1: Load the Dataset

We use the Titanic dataset. It is the most documented teaching dataset for EDA.
Every pattern we find here is verifiable against published analysis.

We load it directly from seaborn — no file downloads needed.

In [ ]:
# Load Titanic dataset from seaborn (internet connection required)
df = sns.load_dataset('titanic')

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print()
print("First 5 rows:")
df.head()

In [ ]:
# Step 1a: Initial audit — always run this before anything else
# This gives you the full picture in 3 commands

print("=" * 50)
print("COLUMN DATA TYPES")
print("=" * 50)
print(df.dtypes)

print()
print("=" * 50)
print("MISSING VALUE COUNT PER COLUMN")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_report[missing_report['Missing Count'] > 0])

print()
print("=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
df.describe()

## Step 2: Univariate Analysis

Univariate means one column at a time. You are asking:
- What is the distribution of this column?
- Is it skewed?
- Are there unexpected values?

You do this for every column before looking at relationships.

In [ ]:
# Univariate analysis for numerical columns
# We look at Age and Fare — two key numerical features in this dataset

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
axes[0].hist(df['age'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['age'].mean(), color='red', linestyle='--', label=f"Mean: {df['age'].mean():.1f}")
axes[0].axvline(df['age'].median(), color='orange', linestyle='--', label=f"Median: {df['age'].median():.1f}")
axes[0].set_title('Age Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# Fare distribution (right-skewed, common in real data)
axes[1].hist(df['fare'].dropna(), bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(df['fare'].mean(), color='red', linestyle='--', label=f"Mean: {df['fare'].mean():.1f}")
axes[1].axvline(df['fare'].median(), color='orange', linestyle='--', label=f"Median: {df['fare'].median():.1f}")
axes[1].set_title('Fare Distribution (Right-Skewed)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Fare')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Univariate Analysis: Numerical Columns', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Fare skewness: {df['fare'].skew():.2f}")
print("Skewness > 1 means right-skewed. Log transform recommended before using Fare in linear models.")

In [ ]:
# Univariate analysis for categorical columns
# value_counts() is your primary tool here

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Passenger class
pclass_counts = df['pclass'].value_counts().sort_index()
axes[0].bar(['1st Class', '2nd Class', '3rd Class'], pclass_counts.values,
            color=['#1a78c2', '#43a6db', '#7bc8f6'], edgecolor='white')
axes[0].set_title('Passenger Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(pclass_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Sex
sex_counts = df['sex'].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values,
            color=['#1a78c2', '#e87e8a'], edgecolor='white')
axes[1].set_title('Sex Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(sex_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Embarked port
emb_counts = df['embarked'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db']
axes[2].bar(emb_counts.index, emb_counts.values, color=colors, edgecolor='white')
axes[2].set_title('Port of Embarkation', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Count')
for i, v in enumerate(emb_counts.values):
    axes[2].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.suptitle('Univariate Analysis: Categorical Columns', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Observations:")
print(f"  - 3rd class passengers: {pclass_counts[3]} ({pclass_counts[3]/len(df)*100:.1f}% of total)")
print(f"  - Male passengers: {sex_counts['male']} ({sex_counts['male']/len(df)*100:.1f}% of total)")
print(f"  - Southampton departures: {emb_counts.get('S', 0)} ({emb_counts.get('S', 0)/len(df)*100:.1f}% of total)")

## Step 3: Target Variable Analysis + Class Imbalance Check

This is the step most beginners skip and it costs them in production.

**The accuracy paradox:** A churn prediction model with 94% accuracy sounds great.
If 94% of customers don't churn, a model that predicts "no churn" for everyone
scores 94% accuracy while being completely useless.

Always check your target variable distribution before anything else.

In [ ]:
# Target variable analysis
# For classification: always check class distribution

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
survived_counts = df['survived'].value_counts()
labels = ['Did Not Survive', 'Survived']
colors = ['#E53935', '#43A047']
bars = axes[0].bar(labels, survived_counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Survival Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Passengers')
for bar, count in zip(bars, survived_counts.values):
    pct = count / len(df) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                 f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=11)

# Pie chart for imbalance ratio
axes[1].pie(survived_counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (Pie View)', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: Survival (Class Imbalance Check)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

ratio = survived_counts[0] / survived_counts[1]
print(f"Class ratio (not survived : survived) = {ratio:.2f} : 1")

if ratio > 1.5:
    print("Class imbalance detected. Do NOT use accuracy alone as your evaluation metric.")
    print("Use Precision, Recall, F1-score, and ROC-AUC instead.")
else:
    print("Classes are reasonably balanced. Accuracy is a reliable metric here.")

## Step 4: Bivariate Analysis

Bivariate means looking at the relationship between two columns.
You want to understand how features relate to your target variable.

This is where you find signals that will become features in your model.

In [ ]:
# Bivariate: categorical feature vs target variable
# Survival rate by Pclass, Sex, and Embarked

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Survival rate by Pclass
pclass_survival = df.groupby('pclass')['survived'].mean() * 100
bars = axes[0].bar(['1st Class', '2nd Class', '3rd Class'],
                   pclass_survival.values,
                   color=['#1a78c2', '#43a6db', '#7bc8f6'], edgecolor='white')
axes[0].set_title('Survival Rate by Passenger Class', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_ylim(0, 80)
for bar, val in zip(bars, pclass_survival.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Survival rate by Sex
sex_survival = df.groupby('sex')['survived'].mean() * 100
bars = axes[1].bar(sex_survival.index, sex_survival.values,
                   color=['#1a78c2', '#e87e8a'], edgecolor='white')
axes[1].set_title('Survival Rate by Sex', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 90)
for bar, val in zip(bars, sex_survival.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Survival rate by Embarked
emb_survival = df.groupby('embarked')['survived'].mean() * 100
colors = ['#2ecc71', '#e74c3c', '#3498db']
bars = axes[2].bar(emb_survival.index, emb_survival.values, color=colors, edgecolor='white')
axes[2].set_title('Survival Rate by Embarkation Port', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Survival Rate (%)')
axes[2].set_ylim(0, 75)
for bar, val in zip(bars, emb_survival.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Bivariate Analysis: Feature vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key findings from bivariate analysis:")
print(f"  - 1st class survival rate: {pclass_survival[1]:.1f}% vs 3rd class: {pclass_survival[3]:.1f}%")
print(f"  - Female survival rate: {sex_survival['female']:.1f}% vs Male: {sex_survival['male']:.1f}%")
print("  -> Sex and Pclass are strong predictors of survival")

In [ ]:
# Bivariate: numerical feature vs target
# Boxplots show distribution of a numerical feature split by target class

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age by Survived
survived_group = df[df['survived'] == 1]['age'].dropna()
not_survived_group = df[df['survived'] == 0]['age'].dropna()

axes[0].boxplot([not_survived_group, survived_group],
               labels=['Did Not Survive', 'Survived'],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Age Distribution by Survival', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Age')

# Fare by Survived
survived_fare = df[df['survived'] == 1]['fare']
not_survived_fare = df[df['survived'] == 0]['fare']

axes[1].boxplot([not_survived_fare, survived_fare],
               labels=['Did Not Survive', 'Survived'],
               patch_artist=True,
               boxprops=dict(facecolor='lightyellow', color='darkorange'),
               medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Fare Distribution by Survival', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fare (GBP)')

plt.suptitle('Numerical Features vs Target (Boxplots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Findings:")
print(f"  - Median age (survived): {survived_group.median():.1f}  |  Median age (not survived): {not_survived_group.median():.1f}")
print(f"  - Median fare (survived): {survived_fare.median():.1f}  |  Median fare (not survived): {not_survived_fare.median():.1f}")
print("  -> Passengers who paid higher fares had significantly better survival odds")
print("  -> Age alone is a weak predictor (similar medians) but correlates with class")

## Step 5: Multivariate Analysis

Multivariate analysis looks at patterns across multiple columns simultaneously.
The two most important tools here:
1. **Correlation heatmap** — shows linear relationships between all numerical features
2. **Pairplot** — scatter plots of every numerical pair, colored by the target variable

In [ ]:
# Correlation heatmap
# Only run on numerical columns
# Correlation coefficient ranges from -1 (perfect negative) to +1 (perfect positive)

numerical_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(9, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Hide upper triangle (duplicate)
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    mask=mask,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap (Lower Triangle)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Print top correlations with target
target_corr = corr_matrix['survived'].drop('survived').abs().sort_values(ascending=False)
print("Features ranked by absolute correlation with 'survived':")
for feat, val in target_corr.items():
    direction = corr_matrix['survived'][feat]
    sign = '+' if direction > 0 else '-'
    print(f"  {feat:<10} {sign}{val:.3f}")

print()
print("Note: pclass shows negative correlation — higher class number (3rd) = lower survival")
print("Fare shows positive correlation — higher fare = higher class = better survival odds")
print("pclass and fare are highly correlated with each other (multicollinearity risk)")

In [ ]:
# Pairplot — scatter matrix colored by target variable
# This is computationally heavier, keep to 4-5 columns max

pairplot_df = df[['survived', 'age', 'fare', 'pclass']].dropna().copy()
pairplot_df['survived'] = pairplot_df['survived'].map({0: 'Not Survived', 1: 'Survived'})

g = sns.pairplot(
    pairplot_df,
    hue='survived',
    palette={'Not Survived': '#E53935', 'Survived': '#43A047'},
    plot_kws={'alpha': 0.5, 's': 30},
    diag_kind='kde',
    height=2.5
)
g.fig.suptitle('Pairplot: Age, Fare, Pclass vs Survival', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("What to look for in a pairplot:")
print("  - Clusters that separate by color = strong feature for classification")
print("  - Diagonal KDE plots show distribution of each feature per class")
print("  - If red and green points completely overlap = feature is not useful")

## Step 6: Missing Value Visualisation

Knowing *which* columns have missing values is not enough.
You need to know *where* they are missing and whether they are missing together.

If Age and Cabin are both missing in the same rows, that is a signal — not random noise.

In [ ]:
# Missing value visualisation using missingno
# White bars = missing, dark bars = present

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Matrix view — each column is a strip, each row is a record
# White line in a column = that record is missing for that column
plt.sca(axes[0])
msno.matrix(df, ax=axes[0], sparkline=False, fontsize=10, color=(0.2, 0.4, 0.7))
axes[0].set_title('Missing Value Matrix (white = missing)', fontsize=12, fontweight='bold')

# Bar view — shows % present for each column
plt.sca(axes[1])
msno.bar(df, ax=axes[1], fontsize=10, color='steelblue')
axes[1].set_title('Column Completeness (% present)', fontsize=12, fontweight='bold')

plt.suptitle('Missing Value Visualisation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summarise missing data
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
print("Columns with missing data:")
for col, count in missing_summary.items():
    pct = count / len(df) * 100
    flag = '!!!' if pct > 50 else ('!!' if pct > 20 else '!')
    print(f"  {flag} {col:<15} {count} missing ({pct:.1f}%)")

## Step 7: The Production Problem — The Accuracy Paradox

This is the most common EDA failure in production ML.

**The scenario:** A data scientist builds a churn prediction model.
They skip EDA, go straight to modelling, and report 94% accuracy.
The model ships. It predicts "no churn" for every customer.
It's still 94% accurate because 94% of customers never churned.
The business runs a retention campaign targeting zero customers.

One EDA cell — `df['churn'].value_counts()` — would have caught this.
This is documented in Netflix and Google's internal ML guidelines as a mandatory check.

In [ ]:
# Simulate the accuracy paradox
# We create a severely imbalanced dataset and show how accuracy lies

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# Simulate 1000 customers, 6% churn rate (realistic)
np.random.seed(42)
n_customers = 1000
churn = np.random.choice([0, 1], n_customers, p=[0.94, 0.06])  # 94% don't churn

# Features don't matter for this demo — we're showing the metric problem
X = np.random.randn(n_customers, 5)
y = churn

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# "Lazy" model that predicts majority class for everything
lazy_model = DummyClassifier(strategy='most_frequent')
lazy_model.fit(X_train, y_train)
y_pred = lazy_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("=" * 55)
print("THE ACCURACY PARADOX")
print("=" * 55)
print(f"Class distribution in test set: {np.bincount(y_test)}")
print(f"  -> {np.sum(y_test == 0)} non-churn, {np.sum(y_test == 1)} churn")
print()
print(f"Model that predicts 'no churn' for everyone:")
print(f"  Accuracy : {acc*100:.1f}%   <- Looks great!")
print(f"  F1 Score : {f1:.3f}         <- Actually zero")
print()
print("Full report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

print("LESSON: If you had run value_counts() on the target during EDA,")
print("you would have flagged the imbalance BEFORE training, not after shipping.")

## Step 8: Real World EDA — Reproducing the Instacart Insight

Kazanova's winning solution in the 2017 Instacart Kaggle competition found through EDA
that **time since last order** was the strongest predictor of reorder behavior.
The feature wasn't labeled obviously — it required building it from raw timestamps.

Below we reproduce this insight pattern using a simplified version of the problem.

In [ ]:
# Simulate the Instacart EDA pattern
# This shows how EDA can reveal a non-obvious feature

np.random.seed(42)
n = 500

# Simulate: users who ordered recently are more likely to reorder
days_since_last_order = np.random.exponential(15, n).clip(0, 60)

# Reorder probability decreases as days_since increases
reorder_prob = 1 / (1 + np.exp(0.1 * (days_since_last_order - 15)))
reordered = (np.random.random(n) < reorder_prob).astype(int)

order_data = pd.DataFrame({
    'days_since_last_order': days_since_last_order,
    'reordered': reordered
})

# Bin days_since_last_order into groups for easy comparison
bins = [0, 7, 14, 21, 30, 60]
labels = ['0-7d', '8-14d', '15-21d', '22-30d', '31-60d']
order_data['recency_bucket'] = pd.cut(
    order_data['days_since_last_order'],
    bins=bins, labels=labels, include_lowest=True
)

# EDA: reorder rate by recency bucket
reorder_by_recency = order_data.groupby('recency_bucket', observed=True)['reordered'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reorder rate by days since last order
colors = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
bars = axes[0].bar(reorder_by_recency.index, reorder_by_recency.values, color=colors, edgecolor='white')
axes[0].set_title('Reorder Rate by Time Since Last Order', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Reorder Rate (%)')
axes[0].set_xlabel('Days Since Last Order')
for bar, val in zip(bars, reorder_by_recency.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)

# Distribution of days_since
axes[1].hist(order_data['days_since_last_order'], bins=30,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution of Days Since Last Order', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Days Since Last Order')
axes[1].set_ylabel('Number of Users')

plt.suptitle('EDA Pattern: Discovering the Key Predictor (Instacart Approach)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("EDA Finding:")
print(f"  - Users who ordered in the last 7 days: {reorder_by_recency.iloc[0]:.1f}% reorder rate")
print(f"  - Users who last ordered 31-60 days ago: {reorder_by_recency.iloc[4]:.1f}% reorder rate")
print()
print("This is exactly what Kazanova found in the Instacart competition.")
print("The insight came from EDA, not from the model.")
print("The model just quantified what EDA already revealed.")

## Step 9: The EDA Checklist (Use on Every New Dataset)

This is the repeatable process that should run before every ML project.

In [ ]:
# One-stop EDA summary function
# This is a template you can reuse on any dataset

def run_eda_summary(dataframe, target_col=None):
    """
    Runs a structured EDA summary on any dataframe.
    Prints shape, types, missing values, basic stats,
    and optionally analyzes the target column.
    """
    print("=" * 60)
    print("EDA SUMMARY")
    print("=" * 60)

    print(f"\nShape: {dataframe.shape[0]} rows, {dataframe.shape[1]} columns")

    print("\nData Types:")
    type_counts = dataframe.dtypes.value_counts()
    for dtype, count in type_counts.items():
        print(f"  {str(dtype):<15} {count} column(s)")

    print("\nMissing Values:")
    missing = dataframe.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing) == 0:
        print("  No missing values found.")
    else:
        for col, cnt in missing.items():
            pct = cnt / len(dataframe) * 100
            print(f"  {col:<20} {cnt:>5} missing ({pct:.1f}%)")

    print("\nDuplicate rows:", dataframe.duplicated().sum())

    if target_col and target_col in dataframe.columns:
        print(f"\nTarget column '{target_col}' distribution:")
        counts = dataframe[target_col].value_counts()
        for val, cnt in counts.items():
            pct = cnt / len(dataframe) * 100
            print(f"  {str(val):<20} {cnt:>5} ({pct:.1f}%)")

        if dataframe[target_col].dtype in ['int64', 'int32', 'float64']:
            n_classes = counts.max() / counts.min()
            if n_classes > 3:
                print(f"  WARNING: Class imbalance ratio = {n_classes:.1f}:1")
                print("  Use F1, Precision, Recall, AUC — not accuracy alone")

    print("\nNumerical column skewness:")
    num_cols = dataframe.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        skew = dataframe[col].skew()
        flag = " <- Log transform recommended" if abs(skew) > 1 else ""
        print(f"  {col:<20} skew = {skew:>6.2f}{flag}")

    print("\n" + "=" * 60)
    print("END OF EDA SUMMARY")
    print("=" * 60)


# Run on the Titanic dataset
run_eda_summary(df, target_col='survived')

## Practice Exercise

Now you run EDA on a new dataset without guidance.

We use the **tips** dataset (restaurant bill data from seaborn).

Your task:
1. Run `run_eda_summary()` on the tips dataset
2. Find the top 2 features correlated with `tip`
3. Check for class imbalance in the `smoker` column
4. Plot one bivariate chart of your choice
5. Write 3 observations from your EDA in a markdown cell below your code

In [ ]:
# Load the tips dataset
tips = sns.load_dataset('tips')

print("Tips dataset preview:")
print(tips.head())
print()
print("Columns:", tips.columns.tolist())

# YOUR CODE BELOW
# Step 1: Run EDA summary


# Step 2: Correlation with 'tip'


# Step 3: Smoker distribution


# Step 4: Your bivariate chart


**Your 3 EDA Observations:**

1. 
2. 
3. 

In [ ]:
# SOLUTION — Run this after completing the exercise above

# Step 1
run_eda_summary(tips, target_col=None)

# Step 2: Correlation with tip
print("\nCorrelation with 'tip':")
num_tips = tips.select_dtypes(include=[np.number])
print(num_tips.corr()['tip'].drop('tip').sort_values(ascending=False))

# Step 3: Smoker distribution
print("\nSmoker distribution:")
print(tips['smoker'].value_counts())

# Step 4: Bivariate — tip by day
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

day_tips = tips.groupby('day')['tip'].mean().sort_values(ascending=False)
axes[0].bar(day_tips.index, day_tips.values, color='steelblue', edgecolor='white')
axes[0].set_title('Average Tip by Day', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Tip ($)')

axes[1].scatter(tips['total_bill'], tips['tip'], alpha=0.5, color='coral')
axes[1].set_title('Total Bill vs Tip', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Bill ($)')
axes[1].set_ylabel('Tip ($)')

plt.suptitle('Tips Dataset — Practice EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSample observations:")
print("  1. total_bill has the highest correlation with tip (makes sense — bigger bill, bigger tip)")
print("  2. Smoker/non-smoker split is fairly balanced — no class imbalance issue")
print("  3. Sunday generates slightly higher average tips than other days")

## Summary

| Step | What you did | Why it matters |
|------|-------------|----------------|
| Initial audit | Shape, dtypes, missing values | Catch problems before they reach the model |
| Univariate | Distributions, skewness | Find features that need transformation |
| Target analysis | Class distribution | Catch imbalance before choosing metrics |
| Bivariate | Feature vs target | Find strong predictors early |
| Correlation heatmap | Feature-to-feature correlation | Spot multicollinearity |
| Pairplot | All pairs at once | Visual confirmation of relationships |
| Missing value viz | Where and how much | Decide imputation strategy |

**EDA does not produce a model. It produces hypotheses you test with a model.**

The Instacart winning solution found its best feature through EDA.
The accuracy paradox is caught through EDA.
Multicollinearity between pclass and fare — also EDA.

---

**Day 5 tomorrow: Missing Data**

MCAR, MAR, MNAR — three types that require three completely different strategies.
The kind of mistake that introduces silent bias into production healthcare models.

GitHub repo: https://github.com/VaishnaviJagtap18/42-Days-0f-ML-Challenge